<a href="https://colab.research.google.com/github/AbyssWatcher07/BigData26_B_2311532021_StanleyLeonardDarmali/blob/main/Praktikum04/BD_B_P04_2311532021_StanleyLeonardDarmali.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### K-1. Menyiapkan Lingkungan dan Data Contoh
Kita menyiapkan environment dengan menginstal pustaka utama seperti duckdb, networkx, shapely, tinydb, dan deltalake, lalu menyambungkan Google Drive untuk memuat data *trips_bersih* dari praktikum sebelumnya. Fungsi `ukur()` juga didefinisikan ulang untuk mencatat waktu eksekusi setiap proses.


In [3]:
!pip install -q duckdb networkx shapely tinydb "deltalake>=1.0"
from google.colab import drive
drive.mount("/content/drive")

DIR_P3 = "/content/drive/MyDrive/BigData/Praktikum03"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum04"

# Needed to unzip trips_bersih.zip and then rename the resulting trips_bersih (Unzipped Files) folder to trips_bersih
TRIPS_BERSIH = f"{DIR_P3}/trips_bersih"

import os
os.makedirs(DIR_SIMPAN, exist_ok=True)
assert os.path.exists(TRIPS_BERSIH), "Folder trips_bersih/ tidak ada. Pastikan path P3 sudah benar atau gunakan generator sintetis."

import time, json, sqlite3, random, shutil, glob
import numpy as np, pandas as pd

catatan = []
def ukur(label, fungsi):
    t0 = time.perf_counter()
    hasil = fungsi()
    detik = time.perf_counter() - t0
    catatan.append({"langkah": label, "detik": round(detik, 4)})
    print(f"[{label}] {detik:.4f} s")
    return hasil

# Fix to df = ukur("baca trips_bersih", lambda: pd.read_parquet(TRIPS_BERSIH))
import duckdb
GLOB_TRIPS = f"{TRIPS_BERSIH}/*/*.parquet"
df = ukur("baca trips_bersih", lambda: duckdb.sql(f"SELECT * FROM read_parquet('{GLOB_TRIPS}', hive_partitioning=true)").df())

NAMA_BARU = {
    "tpep_pickup_datetime": "waktu_jemput",
    "PULocationID": "zona_jemput",
    "DOLocationID": "zona_tujuan",
    "borough_naik": "wilayah_jemput",
    "trip_distance": "jarak_mil",
    "total_amount": "total_bayar",
    "tip_amount": "tip",
    "nama_pembayaran": "metode_bayar",
}
df = df.rename(columns=NAMA_BARU)
df["wilayah_jemput"] = df["wilayah_jemput"].astype(str)
print("baris:", f"{len(df):,}", "| kolom:", df.shape[1])

sample = df.sample(n=min(300_000, len(df)), random_state=42).reset_index(drop=True)
sample["id_perjalanan"] = sample.index + 1
sample["waktu_jemput"] = sample["waktu_jemput"].astype(str)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[baca trips_bersih] 3.7349 s
baris: 2,986,910 | kolom: 18


### K-2. Tabel Biasa vs Key-value
*Keputusan Analisis:* Kita akan membandingkan pencarian id perjalanan secara langsung antara database relasional (SQLite tanpa dan dengan indeks) dengan penyimpanan key-value (Kamus Python berbasis teks JSON). Secara teoritis, SQLite dengan indeks dan key-value dictionary akan merespons dalam waktu yang mirip (sangat cepat), sementara pencarian tanpa indeks akan jauh lebih lambat karena harus memeriksa baris per baris (*full scan*). Sebaliknya, untuk tugas agregasi wilayah, SQL diperkirakan menang telak karena ia hanya perlu membaca kolom terstruktur, berbeda dengan key-value yang dipaksa melakukan *parsing* JSON berulang-ulang untuk seluruh baris.


In [4]:
KOLOM = ["id_perjalanan", "waktu_jemput", "zona_jemput", "zona_tujuan",
         "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data = sample[KOLOM].copy()

if os.path.exists("uji.db"):
    os.remove("uji.db")
con = sqlite3.connect("uji.db")
ukur("relasional: muat tabel", lambda: data.to_sql("perjalanan", con, index=False))

def bangun_kv():
    return {f"perjalanan:{r['id_perjalanan']}": json.dumps(r) for r in data.to_dict("records")}

kv = ukur("key-value: bangun dict", bangun_kv)
print("jumlah key:", len(kv))

random.seed(42)
id_uji = random.sample(range(1, len(data) + 1), 200)

def cari_sql():
    cur = con.cursor()
    return [cur.execute("SELECT * FROM perjalanan WHERE id_perjalanan = ?", (i,)).fetchone() for i in id_uji]

def cari_kv():
    return [json.loads(kv[f"perjalanan:{i}"]) for i in id_uji]

r1 = ukur("cari 200 perjalanan - SQL tanpa indeks", cari_sql)
con.execute("CREATE INDEX idx_id ON perjalanan(id_perjalanan)")
r2 = ukur("cari 200 perjalanan - SQL dengan indeks", cari_sql)
r3 = ukur("cari 200 perjalanan - key-value", cari_kv)

print("hasil sama?", r1[0][0] == r2[0][0] == r3[0]["id_perjalanan"])

def hitung_sql():
    return con.execute("""SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
                          FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1""").fetchall()

def hitung_kv():
    total, jumlah = {}, {}
    for teks in kv.values():
        d = json.loads(teks)
        w = d["wilayah_jemput"]
        total[w] = total.get(w, 0) + d["total_bayar"]
        jumlah[w] = jumlah.get(w, 0) + 1
    return sorted([(w, jumlah[w], round(total[w] / jumlah[w], 2)) for w in jumlah],
                  key=lambda x: (-x[1], x[0]))

h1 = ukur("hitung per wilayah - SQL GROUP BY", hitung_sql)
h2 = ukur("hitung per wilayah - key-value (buka semua nilai)", hitung_kv)
print(h1[:3])
print("hasil sama?", h1 == h2)



[relasional: muat tabel] 2.2674 s
[key-value: bangun dict] 9.4175 s
jumlah key: 300000
[cari 200 perjalanan - SQL tanpa indeks] 6.6082 s
[cari 200 perjalanan - SQL dengan indeks] 0.0042 s
[cari 200 perjalanan - key-value] 0.0013 s
hasil sama? True
[hitung per wilayah - SQL GROUP BY] 0.1998 s
[hitung per wilayah - key-value (buka semua nilai)] 1.3900 s
[('Manhattan', 267034, 22.7), ('Queens', 27288, 70.34), ('Unknown', 3787, 39.13)]
hasil sama? True


### K-3. Document: Bentuk Data yang Fleksibel
*Keputusan Analisis:* Kita beralih ke arsitektur dokumen bersarang menggunakan TinyDB untuk menyimpan metadata transaksi. Skema ini sengaja dibuat asimetris di mana objek 'tip' hanya disimpan jika pelanggan membayar via kartu kredit, mensimulasikan fleksibilitas *schemaless* dari NoSQL Document-stores. Saya menduga sistem ini akan mudah menerima injeksi kolom baru secara acak di perjalanan terakhir tanpa menuntut *Alter Table*, namun fleksibilitas ini bisa berisiko meloloskan data kotor jika validasi level-aplikasinya diabaikan.


In [5]:
from tinydb import TinyDB, Query
from tinydb.storages import MemoryStorage

def ke_dokumen(r):
    dok = {
        "id_perjalanan": int(r.id_perjalanan),
        "waktu_jemput": r.waktu_jemput,
        "jemput": {"zona": int(r.zona_jemput), "wilayah": r.wilayah_jemput},
        "tujuan": {"zona": int(r.zona_tujuan)},
        "biaya": {"jarak_mil": float(r.jarak_mil), "total": float(r.total_bayar), "metode_bayar": r.metode_bayar},
    }
    if r.metode_bayar == "Kartu kredit":
        dok["biaya"]["tip"] = float(r.tip)
    return dok

db = TinyDB(storage=MemoryStorage)
tabel = db.table("perjalanan")
docs = [ke_dokumen(r) for r in sample.head(20_000).itertuples()]
tabel.insert_multiple(docs)

print("jumlah dokumen:", len(tabel))
print(json.dumps(tabel.all()[0], indent=2))

Q = Query()
print("jemput di Manhattan & total > 50:", len(tabel.search((Q.jemput.wilayah == "Manhattan") & (Q.biaya.total > 50))))
print("dokumen yang punya field tip:", len(tabel.search(Q.biaya.tip.exists())))
print("dokumen yang TIDAK punya tip:", len(tabel.search(~Q.biaya.tip.exists())))

# Injeksi dokumen dengan kolom baru ("cuaca")
tabel.insert({
    "id_perjalanan": 999_999, "waktu_jemput": "2023-01-15 08:00:00",
    "jemput": {"zona": 161, "wilayah": "Manhattan"}, "tujuan": {"zona": 236},
    "biaya": {"jarak_mil": 1.2, "total": 14.5, "metode_bayar": "Kartu kredit", "tip": 3.0},
    "cuaca": {"hujan": True, "suhu_c": -2.0}
})

print("dokumen yang punya field cuaca:", len(tabel.search(Q.cuaca.exists())))
print("total dokumen sekarang:", len(tabel))



jumlah dokumen: 20000
{
  "id_perjalanan": 1,
  "waktu_jemput": "2023-01-16 10:40:39",
  "jemput": {
    "zona": 48,
    "wilayah": "Manhattan"
  },
  "tujuan": {
    "zona": 164
  },
  "biaya": {
    "jarak_mil": 1.55,
    "total": 19.32,
    "metode_bayar": "Kartu kredit",
    "tip": 3.22
  }
}
jemput di Manhattan & total > 50: 693
dokumen yang punya field tip: 15906
dokumen yang TIDAK punya tip: 4094
dokumen yang punya field cuaca: 1
total dokumen sekarang: 20001


### K-4. Penyimpanan per Kolom: DuckDB pada Parquet
*Keputusan Analisis:* DuckDB akan digunakan untuk membandingkan eksekusi SQL langsung di atas file Parquet dengan eksekusi database baris standar (SQLite). DuckDB diprediksi akan unggul secara signifikan dan memakan waktu kilat berkat arsitekturnya yang berorientasi vektor dan fitur *column pruning* yang hanya membaca kolom yang difilter, menghemat memori dan I/O disk secara masif.


In [6]:
import duckdb

GLOB = f"{TRIPS_BERSIH}/*/*.parquet"
con_d = duckdb.connect()

con_d.sql(f"""
CREATE VIEW perjalanan AS
SELECT tpep_pickup_datetime AS waktu_jemput,
       PULocationID AS zona_jemput,
       DOLocationID AS zona_tujuan,
       borough_naik AS wilayah_jemput,
       trip_distance AS jarak_mil,
       total_amount AS total_bayar,
       nama_pembayaran AS metode_bayar
FROM read_parquet('{GLOB}', hive_partitioning=true)
""")

SQL = """SELECT wilayah_jemput, COUNT(*) AS jumlah, ROUND(AVG(total_bayar), 2) AS rata_bayar
          FROM perjalanan GROUP BY wilayah_jemput ORDER BY jumlah DESC, wilayah_jemput"""
hasil_d = ukur("DuckDB hitung per wilayah (langsung dari Parquet)", lambda: con_d.sql(SQL).df())
print(hasil_d)

KOLOM_S = ["wilayah_jemput", "total_bayar", "jarak_mil", "zona_jemput", "zona_tujuan"]
if os.path.exists("baris.db"):
    os.remove("baris.db")
con_s = sqlite3.connect("baris.db")
ukur("SQLite muat seluruh baris ke tabel", lambda: df[KOLOM_S].to_sql("perjalanan", con_s, index=False))

hasil_s = ukur("SQLite hitung per wilayah", lambda: con_s.execute(
    """SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
       FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1""").fetchall())

print("hasil sama?", [tuple(r) for r in hasil_d.itertuples(index=False)] == [tuple(r) for r in hasil_s])

SQL2 = """SELECT COUNT(*), ROUND(AVG(total_bayar), 2)
            FROM perjalanan WHERE wilayah_jemput = 'Staten Island'"""
print(ukur("DuckDB hanya wilayah Staten Island", lambda: con_d.sql(SQL2).fetchall()))

ukuran_pq = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(TRIPS_BERSIH) for f in fs) / 1024**2
print(f"Parquet ({df.shape[1]} kolom): {ukuran_pq:.1f} MB | SQLite ({len(KOLOM_S)} kolom): {os.path.getsize('baris.db') / 1024**2:.1f} MB")



[DuckDB hitung per wilayah (langsung dari Parquet)] 0.3062 s
               wilayah_jemput   jumlah  rata_bayar
0                   Manhattan  2659609       22.72
1                      Queens   270315       70.46
2                     Unknown    37609       39.24
3                    Brooklyn    15724       32.72
4                       Bronx     3074       34.44
5  __HIVE_DEFAULT_PARTITION__      340       74.72
6               Staten Island      211       74.07
7                         EWR       28      106.41
[SQLite muat seluruh baris ke tabel] 15.2919 s
[SQLite hitung per wilayah] 3.0897 s
hasil sama? True
[DuckDB hanya wilayah Staten Island] 0.0428 s
[(211, 74.07)]
Parquet (18 kolom): 66.5 MB | SQLite (5 kolom): 114.2 MB


### K-5. Graph: Pertanyaan tentang Keterhubungan
*Keputusan Analisis:* Graph berbasis *NetworkX* akan dibangun untuk menganalisis topologi rute asal dan rute tujuan taksi. Arus perjalanan di bawah 50 trip per kombinasi sengaja diabaikan/di-drop sebagai noise pembersihan. SQL secara teoretis bisa menghitung *self-join* (jarak multi-node), tapi eksekusinya diprediksi lebih bertele-tele secara query dibandingkan pencarian kedalaman Graph, meski mungkin waktu eksekusi di dataset kecil ini akan imbang.


In [7]:
import networkx as nx

arus = ukur("hitung arus antar zona", lambda: con_d.sql("""
    SELECT zona_jemput AS asal, zona_tujuan AS tujuan, COUNT(*) AS jumlah
    FROM perjalanan GROUP BY 1, 2""").df())

G = nx.DiGraph()
for r in arus.itertuples(index=False):
    G.add_edge(int(r.asal), int(r.tujuan), jumlah=int(r.jumlah))

print("node (zona):", G.number_of_nodes(), " | edge (arus):", G.number_of_edges())

top_graf = sorted(G.edges(data=True), key=lambda e: (-e[2]["jumlah"], e[0], e[1]))[:5]
print("graph:", [(a, b, d["jumlah"]) for a, b, d in top_graf])

top_sql = arus.sort_values(["jumlah", "asal", "tujuan"], ascending=[False, True, True]).head(5)
print("SQL  :", list(top_sql.itertuples(index=False, name=None)))

populer = sorted(G.in_degree(weight="jumlah"), key=lambda kv: (-kv[1], kv[0]))[:5]
print("zona tujuan terpopuler:", populer)

MIN = 50
zona_awal = populer[0][0]

def dua_langkah_graph():
    hasil = set()
    for b, d1 in G[zona_awal].items():
        if d1["jumlah"] < MIN:
            continue
        for c, d2 in G[b].items():
            if d2["jumlah"] >= MIN and c != zona_awal:
                hasil.add(c)
    return hasil

con_d.register("arus", arus)
def dua_langkah_sql():
    return set(con_d.sql(f"""
        SELECT DISTINCT a2.tujuan
        FROM arus a1 JOIN arus a2 ON a1.tujuan = a2.asal
        WHERE a1.asal = {zona_awal} AND a1.jumlah >= {MIN}
        AND a2.jumlah >= {MIN} AND a2.tujuan <> {zona_awal}""").df()["tujuan"])

hg = ukur("2 langkah - graph (NetworkX)", dua_langkah_graph)
hs = ukur("2 langkah - SQL self-join (DuckDB)", dua_langkah_sql)

print("zona awal:", zona_awal, "| terjangkau (graph):", len(hg), "| (SQL):", len(hs), " | sama?", hg == hs)

H = nx.DiGraph([(a, b) for a, b, d in G.edges(data=True) if d["jumlah"] >= MIN])
for k in (1, 2, 3, 4):
    n = len(nx.single_source_shortest_path_length(H, zona_awal, cutoff=k)) - 1
    print(f"dalam <= {k} langkah: {n} zona")



[hitung arus antar zona] 0.3790 s
node (zona): 262  | edge (arus): 21583
graph: [(237, 236, 22065), (236, 237, 18810), (236, 236, 14110), (237, 237, 13753), (264, 264, 13562)]
SQL  : [(237, 236, 22065), (236, 237, 18810), (236, 236, 14110), (237, 237, 13753), (264, 264, 13562)]
zona tujuan terpopuler: [(236, 144209), (237, 130369), (161, 113991), (230, 87706), (170, 87110)]
[2 langkah - graph (NetworkX)] 0.0062 s
[2 langkah - SQL self-join (DuckDB)] 0.0112 s
zona awal: 236 | terjangkau (graph): 214 | (SQL): 214  | sama? True
dalam <= 1 langkah: 74 zona
dalam <= 2 langkah: 214 zona
dalam <= 3 langkah: 214 zona
dalam <= 4 langkah: 214 zona


### K-6. Indeks Spasial
*Keputusan Analisis:* Titik koordinat (lintang-bujur) sintetis akan dibangun karena data TLC modern tidak memuat koordinat publik. Kita akan membandingkan pencarian lokasi di dalam kotak target secara konvensional (iterasi loop murni) vs filtering *NumPy array* vs *STRtree Spatial Indexing*. STRtree diyakini akan mempercepat pencarian secara drastis dibandingkan loop Python yang akan terkendala lambatnya iterasi objek primitif, namun rasio kenaikan kecepatannya terhadap NumPy hanya akan moderat di ukuran kotak yang lumayan besar.


In [8]:
from shapely import STRtree, box, points

rng = np.random.default_rng(42)
N = 200_000
bujur = rng.normal(-73.97, 0.06, N).clip(-74.25, -73.70)
lintang = rng.normal(40.74, 0.05, N).clip(40.50, 40.92)
titik = points(bujur, lintang)

M = 1000
cx = rng.uniform(-74.05, -73.85, M)
cy = rng.uniform(40.65, 40.85, M)
kotak = [box(x - 0.005, y - 0.005, x + 0.005, y + 0.005) for x, y in zip(cx, cy)]

def cara_loop():
    return [sum(1 for p in titik if k.contains(p)) for k in kotak[:5]]

def cara_numpy():
    hasil = []
    for x, y in zip(cx, cy):
        m = (bujur >= x - 0.005) & (bujur <= x + 0.005) & (lintang >= y - 0.005) & (lintang <= y + 0.005)
        hasil.append(int(m.sum()))
    return hasil

h_loop = ukur("cara 1 - loop Python, 5 kotak", cara_loop)
h_np = ukur("cara 2 - NumPy, 1000 kotak", cara_numpy)

pohon = ukur("bangun indeks STRtree", lambda: STRtree(titik))
h_idx = ukur("cara 3 - STRtree, 1000 kotak", lambda: [len(pohon.query(k, predicate="contains")) for k in kotak])

print("5 kotak pertama - loop:", h_loop, "| STRtree:", h_idx[:5])
print("hasil NumPy dan STRtree sama?", h_np == h_idx, "| loop dan STRtree sama (5 pertama)?", h_loop == h_idx[:5])
print("total titik ditemukan (1000 kotak):", sum(h_idx))



[cara 1 - loop Python, 5 kotak] 0.3537 s
[cara 2 - NumPy, 1000 kotak] 0.5383 s
[bangun indeks STRtree] 0.0779 s
[cara 3 - STRtree, 1000 kotak] 0.3236 s
5 kotak pertama - loop: [1012, 84, 228, 748, 509] | STRtree: [1012, 84, 228, 748, 509]
hasil NumPy dan STRtree sama? True | loop dan STRtree sama (5 pertama)? True
total titik ditemukan (1000 kotak): 417545


### K-7. Delta Lake: Versi, Time Travel, dan Schema Enforcement
*Keputusan Analisis:* Kita akan menimpa dataset Parquet mentah ke format arsitektur Lakehouse *Delta Lake* untuk memvalidasi fungsi keandalan ACID. Modifikasi nilai tarif diubah spesifik menggunakan predikat SQL, lalu injeksi tipe skema ilegal (kolom asing) dilemparkan ke tabel untuk memicu 'SchemaMismatchError'. Ini membuktikan bahwa sistem Delta secara otomatis menolak penyimpangan metadata yang berisiko menodai struktur transaksi.


In [10]:
from deltalake import DeltaTable, write_deltalake

KOLOM_DELTA = ["waktu_jemput", "zona_jemput", "zona_tujuan",
               "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data_delta = df[KOLOM_DELTA].sample(n=min(200_000, len(df)), random_state=42).reset_index(drop=True)

#Fix to PATH_DELTA = f"{DIR_SIMPAN}/trips_delta"
PATH_DELTA = "/content/trips_delta"
if os.path.exists(PATH_DELTA):
    shutil.rmtree(PATH_DELTA)

write_deltalake(PATH_DELTA, data_delta.iloc[:100_000], mode="overwrite")
dt = DeltaTable(PATH_DELTA)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

write_deltalake(PATH_DELTA, data_delta.iloc[100_000:150_000], mode="append")
dt = DeltaTable(PATH_DELTA)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

sekarang = data_delta.iloc[:150_000]
n_koreksi = int(((sekarang["wilayah_jemput"] == "Bronx") & (sekarang["metode_bayar"] == "Tunai")).sum())
print("baris yang akan dikoreksi:", n_koreksi)

dt.update(predicate="wilayah_jemput = 'Bronx' AND metode_bayar = 'Tunai'",
          updates={"total_bayar": "total_bayar + 1.0"})
print("versi:", DeltaTable(PATH_DELTA).version())

v0 = DeltaTable(PATH_DELTA, version=0).to_pandas()
v1 = DeltaTable(PATH_DELTA, version=1).to_pandas()
terbaru = DeltaTable(PATH_DELTA).to_pandas()

print("baris v0:", len(v0), "| v1:", len(v1), " | terbaru:", len(terbaru))
selisih = terbaru["total_bayar"].sum() - v1["total_bayar"].sum()
print("selisih total_bayar (terbaru - v1):", round(selisih, 2), " | jumlah baris dikoreksi:", n_koreksi)

print("\nRiwayat Versi:")
for h in DeltaTable(PATH_DELTA).history():
    print(h.get("version"), h.get("operation"))

print("Log Transaksi JSON:", sorted(os.listdir(f"{PATH_DELTA}/_delta_log")))

tambahan = data_delta.iloc[150_000:150_010].copy()
tambahan["kolom_asing"] = 1
try:
    write_deltalake(PATH_DELTA, tambahan, mode="append")
    print("append kolom asing: BERHASIL (tidak diharapkan)")
except Exception as e:
    print("append kolom asing DITOLAK:", type(e).__name__)
print("versi setelah percobaan gagal:", DeltaTable(PATH_DELTA).version())



versi: 0 | baris: 100000
versi: 1 | baris: 150000
baris yang akan dikoreksi: 20
versi: 2
baris v0: 100000 | v1: 150000  | terbaru: 150000
selisih total_bayar (terbaru - v1): 20.0  | jumlah baris dikoreksi: 20

Riwayat Versi:
2 UPDATE
1 WRITE
0 WRITE
Log Transaksi JSON: ['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json']
append kolom asing DITOLAK: SchemaMismatchError
versi setelah percobaan gagal: 2


### K-8. Small File Problem pada Tabel Delta
*Keputusan Analisis:* Sebanyak 20 injeksi kecil (ukuran 500 baris) akan sengaja dilakukan guna mensimulasikan sindrom *small-file* yang menurunkan kapabilitas I/O. Setelah itu, metode *Compaction* (Vacuum logikal) akan diaktifkan untuk melemburkan puluhan metadata file pecah-pecah tersebut ke dalam satu berkas Parquet besar yang padat dan efisien, sementara file sisa lamanya di-*dry run* agar terlihat oleh VACUUM retention tanpa benar-benar merusak kemampuan *time-travel*.


In [11]:
dt = DeltaTable(PATH_DELTA)
print("file data aktif sebelum:", len(dt.file_uris()), "| versi:", dt.version())

for i in range(20):
    potongan = data_delta.iloc[150_000 + i * 500 : 150_000 + (i + 1) * 500]
    write_deltalake(PATH_DELTA, potongan, mode="append")

dt = DeltaTable(PATH_DELTA)
print("file data aktif setelah 20 append kecil:", len(dt.file_uris()), "| versi:", dt.version(), "| baris:", len(dt.to_pandas()))

t0 = time.perf_counter(); DeltaTable(PATH_DELTA).to_pandas(); t_sebelum = time.perf_counter() - t0

hasil = dt.optimize.compact()
print("hasil compact:", {k: hasil[k] for k in ("numFilesAdded", "numFilesRemoved")})

dt = DeltaTable(PATH_DELTA)
t0 = time.perf_counter(); _ = DeltaTable(PATH_DELTA).to_pandas(); t_sesudah = time.perf_counter() - t0

print("file data aktif setelah compact:", len(dt.file_uris()), "| versi:", dt.version(), "| baris:", len(dt.to_pandas()))
print(f"waktu baca sebelum compact: {t_sebelum:.4f} s | sesudah: {t_sesudah:.4f} s")

print("file parquet di disk:", len(glob.glob(f"{PATH_DELTA}/*.parquet")))
kandidat = dt.vacuum(retention_hours=0, enforce_retention_duration=False, dry_run=True)
print("file yang akan dihapus VACUUM (dry run):", len(kandidat))

# Fix to zip to Google Drive
shutil.make_archive(f"{DIR_SIMPAN}/trips_delta", "zip", "/content/trips_delta")



file data aktif sebelum: 1 | versi: 2
file data aktif setelah 20 append kecil: 21 | versi: 22 | baris: 160000
hasil compact: {'numFilesAdded': 1, 'numFilesRemoved': 21}
file data aktif setelah compact: 1 | versi: 23 | baris: 160000
waktu baca sebelum compact: 0.0822 s | sesudah: 0.0503 s
file parquet di disk: 24
file yang akan dihapus VACUUM (dry run): 23


'/content/drive/MyDrive/BigData/Praktikum04/trips_delta.zip'

### K-9. Merangkum dan Memilih
Seluruh catatan parameter benchmark pengujian disimpan ke dalam kerangka benchmark CSV.


In [12]:
rekap = pd.DataFrame(catatan)
rekap.to_csv(f"{DIR_SIMPAN}/benchmark_nosql.csv", index=False)
print(rekap.to_string(index=False))



                                          langkah   detik
                                baca trips_bersih  3.7349
                           relasional: muat tabel  2.2674
                           key-value: bangun dict  9.4175
           cari 200 perjalanan - SQL tanpa indeks  6.6082
          cari 200 perjalanan - SQL dengan indeks  0.0042
                  cari 200 perjalanan - key-value  0.0013
                hitung per wilayah - SQL GROUP BY  0.1998
hitung per wilayah - key-value (buka semua nilai)  1.3900
DuckDB hitung per wilayah (langsung dari Parquet)  0.3062
               SQLite muat seluruh baris ke tabel 15.2919
                        SQLite hitung per wilayah  3.0897
               DuckDB hanya wilayah Staten Island  0.0428
                           hitung arus antar zona  0.3790
                     2 langkah - graph (NetworkX)  0.0062
               2 langkah - SQL self-join (DuckDB)  0.0112
                    cara 1 - loop Python, 5 kotak  0.3537
              

### Q. Latihan 1-6
*Keputusan Analisis Latihan Mandiri:* Kode Latihan digabung di satu blok sel. Modifikasi dilakukan terhadap batas uji iterasi hingga menembus 2.000 id raw (L1) agar mengkalkulasi waktu rata-rata eksekusi, menjalankan silang-perbandingan multi-kolom Parquet dan NetworkX in_degrees (L2-L4), menyesuaikan toleransi limit spasi 0.02 vs 0.001 derajat (L5), dan mengaktifkan argumen *Restore* pada Lakehouse untuk mendemonstrasikan versi delta-timetravel rollback logaritma secara struktural.


In [13]:
# Q1
random.seed(42)
id_uji_2000 = random.sample(range(1, len(data) + 1), 2000)

def cari_sql_2000():
    cur = con.cursor()
    return [cur.execute("SELECT * FROM perjalanan WHERE id_perjalanan = ?", (i,)).fetchone() for i in id_uji_2000]

def cari_kv_2000():
    return [json.loads(kv[f"perjalanan:{i}"]) for i in id_uji_2000]

t_sql_2000 = ukur("Latihan 1: SQL indeks (2000 id)", cari_sql_2000)
t_kv_2000 = ukur("Latihan 1: KV (2000 id)", cari_kv_2000)

print(f"Rata-rata cari SQL: {catatan[-2]['detik']/2000 * 1e6:.1f} mikrodetik")
print(f"Rata-rata cari KV : {catatan[-1]['detik']/2000 * 1e6:.1f} mikrodetik")

# Q2
q2_tinydb = len(tabel.search((Query().tujuan.zona == 46) & (Query().biaya.metode_bayar == "Tunai")))
q2_sql = con.execute("SELECT COUNT(*) FROM perjalanan WHERE zona_tujuan=46 AND metode_bayar='Tunai'").fetchone()[0]
print(f"\nLatihan 2: TinyDB = {q2_tinydb}, SQL = {q2_sql} | Sama? {q2_tinydb == q2_sql}")

# Q3
t_sum1 = ukur("Latihan 3: DuckDB SUM 1 Kolom", lambda: con_d.sql("SELECT SUM(total_bayar) FROM perjalanan").fetchall())
t_sum2 = ukur("Latihan 3: DuckDB SUM 4 Kolom", lambda: con_d.sql("SELECT SUM(total_bayar), SUM(jarak_mil), SUM(zona_jemput), SUM(zona_tujuan) FROM perjalanan").fetchall())

# Q4
out_pop = sorted(G.out_degree(weight="jumlah"), key=lambda kv: (-kv[1], kv[0]))[:5]
q4_sql = con_d.sql("SELECT asal, COUNT(DISTINCT tujuan) AS unik_tujuan FROM arus GROUP BY asal ORDER BY unik_tujuan DESC, asal ASC LIMIT 5").df()
print(f"\nLatihan 4: Out-degree Graph teratas: {out_pop[:2]}")
print(f"Latihan 4: SQL DISTINCT Tujuan:\n{q4_sql.head(2)}")

# Q5
def test_spatial(ukuran_derajat, N_kotak=1000):
    cx_s = rng.uniform(-74.05, -73.85, N_kotak)
    cy_s = rng.uniform(40.65, 40.85, N_kotak)
    kotak_s = [box(x - ukuran_derajat, y - ukuran_derajat, x + ukuran_derajat, y + ukuran_derajat) for x, y in zip(cx_s, cy_s)]

    def np_test():
        for x, y in zip(cx_s, cy_s):
            m = (bujur >= x - ukuran_derajat) & (bujur <= x + ukuran_derajat) & (lintang >= y - ukuran_derajat) & (lintang <= y + ukuran_derajat)
            m.sum()

    t_np = ukur(f"Latihan 5: NumPy ({ukuran_derajat} derajat)", np_test)
    t_idx = ukur(f"Latihan 5: STRtree ({ukuran_derajat} derajat)", lambda: [pohon.query(k, predicate="contains") for k in kotak_s])

print("\nUji Spasial 0.02 Derajat:")
test_spatial(0.02)
print("Uji Spasial 0.001 Derajat:")
test_spatial(0.001)

# Q6
print("\nLatihan 6: Restore Delta Table")
dt = DeltaTable(PATH_DELTA)
print("versi sebelum restore:", dt.version(), "| baris:", len(dt.to_pandas()))
dt.restore(1)
dt = DeltaTable(PATH_DELTA)
print("versi setelah restore:", dt.version(), "| baris:", len(dt.to_pandas()))
print([h.get("operation") for h in dt.history()][:3])



[Latihan 1: SQL indeks (2000 id)] 0.0377 s
[Latihan 1: KV (2000 id)] 0.0132 s
Rata-rata cari SQL: 18.8 mikrodetik
Rata-rata cari KV : 6.6 mikrodetik

Latihan 2: TinyDB = 0, SQL = 3 | Sama? False
[Latihan 3: DuckDB SUM 1 Kolom] 0.1999 s
[Latihan 3: DuckDB SUM 4 Kolom] 0.3597 s

Latihan 4: Out-degree Graph teratas: [(132, 152840), (237, 145797)]
Latihan 4: SQL DISTINCT Tujuan:
   asal  unik_tujuan
0   132          260
1   138          254

Uji Spasial 0.02 Derajat:
[Latihan 5: NumPy (0.02 derajat)] 0.5268 s
[Latihan 5: STRtree (0.02 derajat)] 3.0304 s
Uji Spasial 0.001 Derajat:
[Latihan 5: NumPy (0.001 derajat)] 0.5559 s
[Latihan 5: STRtree (0.001 derajat)] 0.0206 s

Latihan 6: Restore Delta Table
versi sebelum restore: 23 | baris: 160000
versi setelah restore: 24 | baris: 150000
['RESTORE', 'OPTIMIZE', 'WRITE']


### R.2 Data Lain, Cara Lain: Pemodelan Entitas Zona
*Keputusan Analisis:* Kita akan mengekstrak entitas independen "zona" (membuang duplikat `zona_jemput` dan `wilayah_jemput`). Tiga metode penyimpanan yang dipilih untuk mendemonstrasikan keunggulan/kelemahan NoSQL adalah: Relasional (SQLite), Document (TinyDB), dan Key-Value (Dictionary).
- **Relasional (SQLite):**
  - *Pertanyaan Mudah:* Menghitung jumlah zona per wilayah (via agregasi `GROUP BY`).
  - *Pertanyaan Canggung:* Menyimpan koordinat spasial batas poligon tiap zona yang ukurannya dinamis (skema relasional kaku terhadap array/objek spasial).
- **Document (TinyDB):**
  - *Pertanyaan Mudah:* Menyimpan metadata asimetris (misal: zona bandara EWR diberi field `jumlah_terminal`, sementara zona biasa tidak punya field tersebut).
  - *Pertanyaan Canggung:* Melakukan operasi JOIN agregasi masif dengan tabel transaksi (harus ditarik manual ke memori aplikasi).
- **Key-Value (Dict):**
  - *Pertanyaan Mudah:* Mengambil nama wilayah dari ID zona 132 secara instan `O(1)` dengan kunci `zona:132`.
  - *Pertanyaan Canggung:* Mencari "Sebutkan semua ID zona di wilayah Queens!" (karena mengharuskan *full table scan* membuka isi seluruh value di dalam database).

--- 1. Relasional (SQLite) ---
[Mudah] Jumlah zona di wilayah Queens: 68

--- 2. Document (TinyDB) ---
[Mudah] Jumlah dokumen zona bandara EWR: 1

--- 3. Key-Value (Dictionary) ---
[Mudah] Pengecekan instan wilayah untuk kunci zona:132 -> Queens


### R.3 Delta Lake Dua Hari (Simulasi Transaksi ACID)
*Keputusan Analisis:* Untuk mendemonstrasikan keandalan format arsitektur Lakehouse, kita akan menyimulasikan transaksi ETL selama dua hari berturut-turut pada folder Delta yang sama.
- **Hari 1:** Injeksi *batch* data awal.
- **Hari 2:** Penambahan (*append*) perjalanan baru, yang langsung disusul oleh operasi koreksi (*update*) pada baris-baris tertentu (misalnya merevisi tarif Bronx).
Keberhasilan Lakehouse akan dibuktikan dengan kemampuannya melakukan *Time Travel* secara sempurna (membaca persis keadaan baris di akhir Hari 1 tanpa terpengaruh update Hari 2), serta kelengkapan berkas `_delta_log/` yang mendokumentasikan log atomiknya.

In [18]:
import shutil
PATH_HARI = "/content/trips_2hari"
if os.path.exists(PATH_HARI):
    shutil.rmtree(PATH_HARI)

print("=== SIMULASI HARI 1 ===")
# Menulis data awal (10.000 baris pertama)
hari1_data = data_delta.iloc[:10000]
write_deltalake(PATH_HARI, hari1_data, mode="overwrite")
dt_hari = DeltaTable(PATH_HARI)
print(f"Akhir Hari 1 | Versi: {dt_hari.version()} | Jumlah baris: {len(dt_hari.to_pandas())}")

print("\n=== SIMULASI HARI 2 ===")
# Menambah data baru (baris 10.000 s.d. 20.000)
hari2_data = data_delta.iloc[10000:20000]
write_deltalake(PATH_HARI, hari2_data, mode="append")

# Mengoreksi total_bayar karena ada bug argometer di wilayah Bronx
dt_hari = DeltaTable(PATH_HARI)
dt_hari.update(predicate="wilayah_jemput = 'Bronx'", updates={"total_bayar": "total_bayar + 2.0"})
print(f"Akhir Hari 2 (Selesai Append & Update) | Versi: {dt_hari.version()} | Jumlah baris: {len(dt_hari.to_pandas())}")

print("\n=== PEMBUKTIAN TIME TRAVEL & LOG ===")
baca_hari1 = DeltaTable(PATH_HARI, version=0).to_pandas()
print(f"Membaca paksa keadaan tabel di Hari 1 (Version 0): {len(baca_hari1)} baris (Data Hari 2 diabaikan dengan aman).")

# Menampilkan file JSON log transaksi
log_files = sorted([f for f in os.listdir(f"{PATH_HARI}/_delta_log/") if f.endswith(".json")])
print("\nIsi folder _delta_log/ (Dokumentasi ACID):")
for log in log_files:
    print(f"- {log}")

=== SIMULASI HARI 1 ===
Akhir Hari 1 | Versi: 0 | Jumlah baris: 10000

=== SIMULASI HARI 2 ===
Akhir Hari 2 (Selesai Append & Update) | Versi: 2 | Jumlah baris: 20000

=== PEMBUKTIAN TIME TRAVEL & LOG ===
Membaca paksa keadaan tabel di Hari 1 (Version 0): 10000 baris (Data Hari 2 diabaikan dengan aman).

Isi folder _delta_log/ (Dokumentasi ACID):
- 00000000000000000000.json
- 00000000000000000001.json
- 00000000000000000002.json
